In [ ]:
%pip uninstall --yes 'keras' 'matplotlib' 'scikit-learn' 'tensorflow'

In [ ]:
import warnings
warnings.simplefilter('ignore')

In [ ]:
import os
import sys
import subprocess

In [ ]:
def set_env(input_archive, temp_dir):

    if not os.path.exists(temp_dir):
        os.makedirs(temp_dir, exist_ok=True)
        
        subprocess.run(['tar', '-xzf', input_archive, '-C', temp_dir], check=True)
    
    subprocess.run([
        sys.executable, 
        '-m', 
        'pip', 
        'install', 
        '--no-index', 
        '--find-links', 
        f'{temp_dir}/wheels', 
        'unsloth', 
        'trl', 
        'vllm', 
        'openai_harmony'
    ], check=True)

In [ ]:
set_env(
    input_archive='/kaggle/input/aimo-3-utils/wheels.tar.gz', 
    temp_dir='/kaggle/tmp/setup'
)

In [ ]:
subprocess.run(['ls', '/kaggle/tmp/setup/tiktoken_encodings'])

In [ ]:
os.environ['TRANSFORMERS_NO_TF'] = '1'
os.environ['TRANSFORMERS_NO_FLAX'] = '1'
os.environ['CUDA_VISIBLE_DEVICES'] = '0'
os.environ['TOKENIZERS_PARALLELISM'] = 'false'
os.environ['TRITON_PTXAS_PATH'] = '/usr/local/cuda/bin/ptxas'
os.environ['TIKTOKEN_ENCODINGS_BASE'] = '/kaggle/tmp/setup/tiktoken_encodings'

In [ ]:
import gc
import re
import math
import time
import queue
import threading
import contextlib
from typing import Optional
from jupyter_client import KernelManager
from collections import Counter, defaultdict
from concurrent.futures import as_completed, ThreadPoolExecutor

import numpy as np
import pandas as pd
import polars as pl

# Configure matplotlib before importing pyplot
import matplotlib
matplotlib.use('Agg')  # Non-interactive backend
matplotlib.rcParams['font.family'] = 'sans-serif'
matplotlib.rcParams['font.sans-serif'] = ['Liberation Sans', 'DejaVu Sans', 'Arial', 'Helvetica', 'sans-serif']
matplotlib.rcParams['pdf.fonttype'] = 42
matplotlib.rcParams['ps.fonttype'] = 42

import matplotlib.pyplot as plt
import matplotlib.cm as cm

from openai import OpenAI

from openai_harmony import (
    HarmonyEncodingName, 
    load_harmony_encoding, 
    SystemContent, 
    ReasoningEffort, 
    ToolNamespaceConfig, 
    Author, 
    Message, 
    Role, 
    TextContent, 
    Conversation
)

from transformers import set_seed
import kaggle_evaluation.aimo_3_inference_server

In [ ]:
class CFG:
    
    system_prompt = (
        'You are an elite mathematical problem solver with expertise at the International '
        'Mathematical Olympiad (IMO) level. Your goal is to find the correct answer through '
        'rigorous mathematical reasoning.\n\n'
        
        '# Problem-Solving Approach:\n'
        '1. UNDERSTAND: Carefully read and rephrase the problem in your own words. '
        'Identify what is given, what needs to be found, and any constraints.\n'
        '2. EXPLORE: Consider multiple solution strategies. Think about relevant theorems, '
        'techniques, patterns, or analogous problems. Don\'t commit to one approach immediately.\n'
        '3. PLAN: Select the most promising approach and outline key steps before executing.\n'
        '4. EXECUTE: Work through your solution methodically. Show all reasoning steps clearly.\n'
        '5. VERIFY: Check your answer by substituting back, testing edge cases, or using '
        'alternative methods. Ensure logical consistency throughout.\n\n'
        
        '# Mathematical Reasoning Principles:\n'
        '- Break complex problems into smaller, manageable sub-problems\n'
        '- Look for patterns, symmetries, and special cases that provide insight\n'
        '- Use concrete examples to build intuition before generalizing\n'
        '- Consider extreme cases and boundary conditions\n'
        '- If stuck, try working backwards from the desired result\n'
        '- Be willing to restart with a different approach if needed\n\n'
        
        '# Verification Requirements:\n'
        '- Cross-check arithmetic and algebraic manipulations\n'
        '- Verify that your solution satisfies all problem constraints\n'
        '- Test your answer with simple cases or special values when possible\n'
        '- Ensure dimensional consistency and reasonableness of the result\n\n'
        
        '# Output Format:\n'
        'The final answer must be a non-negative integer between 0 and 99999.\n'
        'Place your final numerical answer inside \\boxed{}, e.g., \\boxed{42}\n\n'
        
        '# Important - Save Intermediate Answers:\n'
        'Whenever you arrive at a candidate answer (even before full verification), '
        'immediately record it using \\unverified_boxed{}, e.g., \\unverified_boxed{42}.\n'
        'You may write \\unverified_boxed{} multiple times as your answer evolves. '
        'Once you have fully verified the answer, place the final confirmed answer in '
        '\\boxed{} as usual. The \\unverified_boxed{} serves as a safety net in case '
        'you run out of time or tokens before completing verification.\n\n'
        
        'Think step-by-step and show your complete reasoning process. Quality of reasoning '
        'is as important as the final answer.'
    )
    
    tool_prompt = (
        'Use this tool to execute Python code for:\n'
        '- Complex calculations that would be error-prone by hand\n'
        '- Numerical verification of analytical results\n'
        '- Generating examples or testing conjectures\n'
        '- Visualizing problem structure when helpful\n'
        '- Brute-force verification for small cases\n\n'
        
        'The environment is a stateful Jupyter notebook. Code persists between executions.\n'
        'Always use print() to display results. Write clear, well-commented code.\n\n'
        
        'Remember: Code should support your mathematical reasoning, not replace it. '
        'Explain what you\'re computing and why before running code.'
    )
    
    preference_prompt = (
        'You have access to `math`, `numpy`, and `sympy` for:\n\n'
        
        '# Symbolic Computation (sympy):\n'
        '- Algebraic manipulation and simplification\n'
        '- Solving equations and systems of equations\n'
        '- Symbolic differentiation and integration\n'
        '- Number theory functions (primes, divisors, modular arithmetic)\n'
        '- Polynomial operations and factorization\n'
        '- Working with mathematical expressions symbolically\n\n'
        
        '# Numerical Computation (numpy):\n'
        '- Array operations and linear algebra\n'
        '- Efficient numerical calculations for large datasets\n'
        '- Matrix operations and eigenvalue problems\n'
        '- Statistical computations\n\n'
        
        '# Mathematical Functions (math):\n'
        '- Standard mathematical functions (trig, log, exp)\n'
        '- Constants like pi and e\n'
        '- Basic operations for single values\n\n'
        
        'Best Practices:\n'
        '- Use sympy for exact symbolic answers when possible\n'
        '- Use numpy for numerical verification and large-scale computation\n'
        '- Use mpmath for high-precision arithmetic\n'
        '- Combine symbolic and numerical approaches: derive symbolically, verify numerically\n'
        '- Document your computational strategy clearly\n'
        '- Validate computational results against known cases or theoretical bounds'
    )

    served_model_name = 'gpt-oss'
    model_path = '/kaggle/input/gpt-oss-120b/transformers/default/1'
    
    kv_cache_dtype = 'fp8_e4m3'
    dtype = 'auto'

    high_problem_timeout = 560 # Changed from 900 to 560
    base_problem_timeout = 300

    notebook_limit = 17580 if os.getenv('KAGGLE_IS_COMPETITION_RERUN') else 9800 # 9800 for 28 prblms
    server_timeout = 180

    session_timeout = 960
    jupyter_timeout = 10
    sandbox_timeout = 5

    stream_interval = 200
    context_tokens = 65536
    search_tokens = 1024
    buffer_tokens = 512
    batch_size = 256
    early_stop = 4 
    attempts = 8  
    workers = 16
    turns = 128
    seed = 42

    gpu_memory_utilization = 0.96
    temperature = 1.0
    min_p = 0.02

    # DeepConf parameters
    top_logprobs = 20           # Number of top logprobs to request for confidence calculation
    
    # Sliding window group confidence parameters (per DeepConf paper)
    group_window_size = 1024    # Window size for group confidence (n tokens)
    group_step_size = 256       # Step size for sliding window
    bottom_percent = 0.10       # Bottom 10% for C_bottom-10 calculation
    
    # Confidence filtering (currently disabled)
    conf_filter_percent = 0.90  # Keep top 90% confident traces
    
    # Output directories
    plot_dir = '/kaggle/working/confidence_plots'
    token_conf_dir = '/kaggle/working/token_confidences'  # Directory for token confidence CSVs
    reasoning_dir = '/kaggle/working/reasoning_traces'    # Directory for full reasoning trace CSVs

In [ ]:
set_seed(CFG.seed)

In [ ]:
class AIMO3Template:

    def __init__(self):

        pass

    def get_system_content(self, system_prompt: str, tool_config: ToolNamespaceConfig) -> SystemContent:

        return (
            SystemContent.new()
            .with_model_identity(system_prompt)
            .with_reasoning_effort(reasoning_effort=ReasoningEffort.HIGH)
            .with_tools(tool_config)
        )

    def apply_chat_template(
        self, 
        system_prompt: str, 
        user_prompt: str, 
        tool_config: ToolNamespaceConfig
    ) -> list[Message]:

        system_content = self.get_system_content(system_prompt, tool_config)        
        system_message = Message.from_role_and_content(Role.SYSTEM, system_content)

        user_message = Message.from_role_and_content(Role.USER, user_prompt)

        return [system_message, user_message]

In [ ]:
class AIMO3Sandbox:

    _port_lock = threading.Lock()
    _next_port = 50000

    @classmethod
    def _get_next_ports(cls, count: int = 5) -> list[int]:

        with cls._port_lock:
            ports = list(range(cls._next_port, cls._next_port + count))
            cls._next_port += count

            return ports

    def __init__(self, timeout: float):

        self._default_timeout = timeout
        self._owns_kernel = False
        self._client = None
        self._km = None
        
        ports = self._get_next_ports(5)

        env = os.environ.copy()
        env['PYDEVD_DISABLE_FILE_VALIDATION'] = '1'
        env['PYDEVD_WARN_EVALUATION_TIMEOUT'] = '0'
        env['JUPYTER_PLATFORM_DIRS'] = '1'
        env['PYTHONWARNINGS'] = 'ignore'
        env['MPLBACKEND'] = 'Agg'

        self._km = KernelManager()
        self._km.shell_port = ports[0]
        self._km.iopub_port = ports[1]
        self._km.stdin_port = ports[2]
        self._km.hb_port = ports[3]
        self._km.control_port = ports[4]

        self._km.start_kernel(env=env, extra_arguments=['--Application.log_level=CRITICAL'])

        self._client = self._km.blocking_client()
        self._client.start_channels()
        self._client.wait_for_ready(timeout=self._default_timeout)
        self._owns_kernel = True

        self.execute(
            'import math\n'
            'import sympy\n'
            'import itertools\n'
            'import collections\n'
            'import numpy as np\n'
            'import mpmath\n'
            'mpmath.mp.dps = 64\n'
        )

    def _format_error(self, traceback: list[str]) -> str:

        clean_lines = []

        for frame in traceback:
            clean_frame = re.sub(r'\x1b\[[0-9;]*m', '', frame)

            if 'File "' in clean_frame and 'ipython-input' not in clean_frame:
                continue

            clean_lines.append(clean_frame)

        return ''.join(clean_lines)

    def execute(self, code: str, timeout: float | None = None) -> str:

        client = self._client
        effective_timeout = timeout or self._default_timeout
        
        msg_id = client.execute(
            code, 
            store_history=True, 
            allow_stdin=False, 
            stop_on_error=False
        )

        stdout_parts = []
        stderr_parts = []
        
        start_time = time.time()

        while True:
            elapsed = time.time() - start_time

            if elapsed > effective_timeout:
                self._km.interrupt_kernel()

                return f'[ERROR] Execution timed out after {effective_timeout} seconds'

            try:
                msg = client.get_iopub_msg(timeout=1.0)

            except queue.Empty:
                continue

            if msg.get('parent_header', {}).get('msg_id') != msg_id:
                continue

            msg_type = msg.get('msg_type')
            content = msg.get('content', {})

            if msg_type == 'stream':
                text = content.get('text', '')

                if content.get('name') == 'stdout':
                    stdout_parts.append(text)

                else:
                    stderr_parts.append(text)

            elif msg_type == 'error':
                traceback_list = content.get('traceback', [])

                stderr_parts.append(self._format_error(traceback_list))

            elif msg_type in {'execute_result', 'display_data'}:
                data = content.get('data', {})
                text = data.get('text/plain')

                if text:
                    stdout_parts.append(text if text.endswith('\n') else f'{text}\n')

            elif msg_type == 'status':
                if content.get('execution_state') == 'idle':
                    break

        stdout = ''.join(stdout_parts)
        stderr = ''.join(stderr_parts)

        if stderr:
            return f'{stdout.rstrip()}\n{stderr}' if stdout else stderr

        return stdout if stdout.strip() else '[WARN] No output. Use print() to see results.'

    def close(self):

        with contextlib.suppress(Exception):
            if self._client:
                self._client.stop_channels()

        if self._owns_kernel and self._km is not None:
            with contextlib.suppress(Exception):
                self._km.shutdown_kernel(now=True)

            with contextlib.suppress(Exception):
                self._km.cleanup_resources()

    def reset(self):

        self.execute('%reset -f')
        self.execute('import gc; gc.collect()')

        self.execute(
            'import math\n'
            'import sympy\n'
            'import itertools\n'
            'import collections\n'
            'import numpy as np\n'
            'import mpmath\n'
            'mpmath.mp.dps = 64\n'
        )

    def __del__(self):

        self.close()

In [ ]:
class AIMO3Tool:

    def __init__(self, local_jupyter_timeout: float, tool_prompt: str, sandbox=None):

        self._local_jupyter_timeout = local_jupyter_timeout
        self._tool_prompt = tool_prompt
        self._jupyter_session = sandbox
        
        self._owns_session = sandbox is None
        
        self._execution_lock = threading.Lock()
        self._init_lock = threading.Lock()

    def _ensure_session(self):

        if self._jupyter_session is None:
            with self._init_lock:
                if self._jupyter_session is None:
                    self._jupyter_session = AIMO3Sandbox(timeout=self._local_jupyter_timeout)

    def _ensure_last_print(self, code: str) -> str:

        lines = code.strip().split('\n')

        if not lines:
            return code

        last_line = lines[-1].strip()

        if 'print' in last_line or 'import' in last_line:
            return code

        if not last_line:
            return code

        if last_line.startswith('#'):
            return code

        lines[-1] = 'print(' + last_line + ')'

        return '\n'.join(lines)

    @property
    def instruction(self) -> str:

        return self._tool_prompt

    @property
    def tool_config(self) -> ToolNamespaceConfig:

        return ToolNamespaceConfig(
            name='python', 
            description=self.instruction, 
            tools=[]
        )

    def _make_response(self, output: str, channel: str | None = None) -> Message:

        content = TextContent(text=output)
        author = Author(role=Role.TOOL, name='python')
        message = Message(author=author, content=[content]).with_recipient('assistant')

        if channel:
            message = message.with_channel(channel)

        return message

    def process_sync_plus(self, message: Message) -> list[Message]:

        self._ensure_session()
        raw_script = message.content[0].text
        final_script = self._ensure_last_print(raw_script)

        with self._execution_lock:
            try:
                output = self._jupyter_session.execute(final_script)

            except TimeoutError as exc:
                output = f'[ERROR] {exc}'

        return [self._make_response(output, channel=message.channel)]

    def close(self):

        if self._jupyter_session is not None:
            if self._owns_session:
                self._jupyter_session.close()

            self._jupyter_session = None

    def __del__(self):

        self.close()

In [ ]:
class AIMO3Solver:

    def __init__(self, cfg, port: int = 8000):

        self.cfg = cfg
        self.port = port
        self.base_url = f'http://0.0.0.0:{port}/v1'
        self.api_key = 'sk-local'
        self.template = AIMO3Template()
        self.encoding = load_harmony_encoding(HarmonyEncodingName.HARMONY_GPT_OSS)
        self.stop_token_ids = self.encoding.stop_tokens_for_assistant_actions()
        
        # Create output directories if needed (only for local validation)
        if not os.getenv('KAGGLE_IS_COMPETITION_RERUN'):
            os.makedirs(self.cfg.plot_dir, exist_ok=True)
            os.makedirs(self.cfg.token_conf_dir, exist_ok=True)
            os.makedirs(self.cfg.reasoning_dir, exist_ok=True)

        self._preload_model_weights()
        
        self.server_process = self._start_server()

        self.client = OpenAI(
            base_url=self.base_url, 
            api_key=self.api_key, 
            timeout=self.cfg.session_timeout
        )

        self._wait_for_server()
        self._initialize_kernels()

        self.notebook_start_time = time.time()
        self.problems_remaining = 50
        self.problem_counter = 0  # For naming plot files

    def _preload_model_weights(self) -> None:

        print(f'Loading model weights from {self.cfg.model_path} into OS Page Cache...')
        start_time = time.time()
        
        files_to_load = []
        total_size = 0

        for root, _, files in os.walk(self.cfg.model_path):
            for file_name in files:
                file_path = os.path.join(root, file_name)

                if os.path.isfile(file_path):
                    files_to_load.append(file_path)
                    total_size += os.path.getsize(file_path)

        def _read_file(path: str) -> None:

            with open(path, 'rb') as file_object:
                while file_object.read(1024 * 1024 * 1024):
                    pass

        with ThreadPoolExecutor(max_workers=self.cfg.workers) as executor:
            list(executor.map(_read_file, files_to_load))

        elapsed = time.time() - start_time
        print(f'Processed {len(files_to_load)} files ({total_size / 1e9:.2f} GB) in {elapsed:.2f} seconds.\n')

    def _start_server(self) -> subprocess.Popen:

        cmd = [
            sys.executable, 
            '-m', 
            'vllm.entrypoints.openai.api_server', 
            '--seed', 
            str(self.cfg.seed), 
            '--model', 
            self.cfg.model_path, 
            '--served-model-name', 
            self.cfg.served_model_name, 
            '--tensor-parallel-size', 
            '1', 
            '--max-num-seqs', 
            str(self.cfg.batch_size), 
            '--gpu-memory-utilization', 
            str(self.cfg.gpu_memory_utilization), 
            '--host', 
            '0.0.0.0', 
            '--port', 
            str(self.port), 
            '--dtype', 
            self.cfg.dtype, 
            '--kv-cache-dtype', 
            self.cfg.kv_cache_dtype, 
            '--max-model-len', 
            str(self.cfg.context_tokens), 
            '--stream-interval', 
            str(self.cfg.stream_interval), 
            '--async-scheduling', 
            '--enable-prefix-caching'
        ]

        self.log_file = open('vllm_server.log', 'w')

        return subprocess.Popen(
            cmd, 
            stdout=self.log_file, 
            stderr=subprocess.STDOUT, 
            start_new_session=True
        )

    def _wait_for_server(self):

        print('Waiting for vLLM server...')
        start_time = time.time()

        for _ in range(self.cfg.server_timeout):
            return_code = self.server_process.poll()

            if return_code is not None:
                self.log_file.flush()

                with open('vllm_server.log', 'r') as log_file:
                    logs = log_file.read()

                raise RuntimeError(f'Server died with code {return_code}. Full logs:\n{logs}\n')

            try:
                self.client.models.list()
                elapsed = time.time() - start_time
                print(f'Server is ready (took {elapsed:.2f} seconds).\n')

                return

            except Exception:
                time.sleep(1)

        raise RuntimeError('Server failed to start (timeout).\n')

    def _initialize_kernels(self) -> None:

        print(f'Initializing {self.cfg.workers} persistent Jupyter kernels...')
        start_time = time.time()

        self.sandbox_pool = queue.Queue()

        def _create_sandbox():
            
            return AIMO3Sandbox(timeout=self.cfg.jupyter_timeout)

        with ThreadPoolExecutor(max_workers=self.cfg.workers) as executor:
            futures = [executor.submit(_create_sandbox) for _ in range(self.cfg.workers)]

            for future in as_completed(futures):
                self.sandbox_pool.put(future.result())

        elapsed = time.time() - start_time
        print(f'Kernels initialized in {elapsed:.2f} seconds.\n')

    def _scan_for_answer(self, text: str) -> int | None:

        pattern = r'\\boxed\s*\{\s*([0-9,]+)\s*\}'
        matches = re.findall(pattern, text)

        if matches:
            try:
                clean_value = matches[-1].replace(',', '')
                value = int(clean_value)

                if 0 <= value <= 99999:
                    return value

            except ValueError:
                pass

        return None

    def _scan_for_unverified_answer(self, text: str) -> int | None:

        pattern = r'\\unverified_boxed\s*\{\s*([0-9,]+)\s*\}'
        matches = re.findall(pattern, text)

        if matches:
            try:
                clean_value = matches[-1].replace(',', '')
                value = int(clean_value)

                if 0 <= value <= 99999:
                    return value

            except ValueError:
                pass

        return None

    def _compute_token_confidences(self, logprobs_list: list) -> list:
        """
        Compute per-token confidences from logprobs.
        
        Per DeepConf paper (arxiv:2508.15260):
        conf_t = -mean(logprobs of ALL top-k tokens)
        
        This measures uncertainty: higher confidence when the model concentrates
        probability mass on fewer tokens.
        """
        token_confs = []
        for token_logprobs in logprobs_list:
            if token_logprobs and len(token_logprobs) > 0:
                valid_lps = [lp for lp in token_logprobs if lp is not None]
                if valid_lps:
                    conf = -sum(valid_lps) / len(valid_lps)
                    token_confs.append(conf)
                else:
                    token_confs.append(0.0)
            else:
                token_confs.append(0.0)
        return token_confs

    def _compute_group_confidences(self, token_confs: list) -> tuple[list, list]:
        """
        Compute sliding window group confidences from token confidences.
        
        Per DeepConf paper:
        C_G_i = (1/|G_i|) * sum(C_t for t in G_i)
        
        where G_i is a sliding window of `group_window_size` tokens.
        
        Returns:
            x_positions: List of token positions (end of each window)
            group_confs: List of group confidence values
        """
        if not token_confs:
            return [], []
        
        window_size = self.cfg.group_window_size
        step_size = self.cfg.group_step_size
        
        x_positions = []
        group_confs = []
        
        # Slide window through token confidences
        start = 0
        while start + window_size <= len(token_confs):
            window = token_confs[start:start + window_size]
            group_conf = sum(window) / len(window)
            
            # X position is the end of the window (token count up to which window is considered)
            x_pos = start + window_size
            x_positions.append(x_pos)
            group_confs.append(group_conf)
            
            start += step_size
        
        # Handle remaining tokens if any (partial window at end)
        if start < len(token_confs) and len(token_confs) - start >= step_size:
            window = token_confs[start:]
            if len(window) > 0:
                group_conf = sum(window) / len(window)
                x_positions.append(len(token_confs))
                group_confs.append(group_conf)
        
        return x_positions, group_confs

    def _compute_bottom10_confidence(self, group_confs: list) -> float:
        """
        Compute Bottom 10% Group Confidence (C_bottom-10) from DeepConf paper.
        
        C_bottom-10(t) = (1/|G_b|) * sum(C_G_j for G_j in G_b)
        
        where G_b is the set of groups with the lowest 10% confidence scores.
        This effectively captures the most problematic reasoning segments.
        """
        if not group_confs:
            return 0.0
        
        # Sort to find bottom 10%
        sorted_confs = sorted(group_confs)
        
        # Calculate how many groups constitute bottom 10%
        num_bottom = max(1, int(len(sorted_confs) * self.cfg.bottom_percent))
        
        # Take the bottom groups
        bottom_confs = sorted_confs[:num_bottom]
        
        return sum(bottom_confs) / len(bottom_confs) if bottom_confs else 0.0

    def _process_attempt(
        self, 
        problem: str, 
        system_prompt: str, 
        attempt_index: int, 
        stop_event: threading.Event, 
        deadline: float
    ) -> dict:

        if stop_event.is_set() or time.time() > deadline:
            return {
                'Attempt': attempt_index + 1, 
                'Answer': None, 
                'Verified': False,
                'Confidence': 0.0,
                'Python Calls': 0, 
                'Python Errors': 0, 
                'Response Length': 0,
                'GroupConfX': [],
                'GroupConfY': [],
                'TokenConfs': [],
                'FullReasoning': '[SKIPPED] Attempt was skipped (stop_event set or deadline passed).'
            }

        local_tool = None
        sandbox = None
        python_calls = 0
        python_errors = 0
        total_tokens = 0
        final_answer = None
        
        # Collect logprobs across ALL turns for the entire trace
        all_logprobs = []
        
        # Accumulate full reasoning trace across all turns
        reasoning_parts = []
        turn_number = 0

        attempt_seed = int(math.pow(self.cfg.seed + attempt_index, 2))

        try:
            sandbox = self.sandbox_pool.get(timeout=self.cfg.sandbox_timeout)

            local_tool = AIMO3Tool(
                local_jupyter_timeout=self.cfg.jupyter_timeout, 
                tool_prompt=self.cfg.tool_prompt, 
                sandbox=sandbox
            )

            encoding = self.encoding
            messages = self.template.apply_chat_template(
                system_prompt, 
                problem, 
                local_tool.tool_config
            )

            conversation = Conversation.from_messages(messages)

            for _ in range(self.cfg.turns):
                if stop_event.is_set() or time.time() > deadline:
                    reasoning_parts.append('\n--- [STOPPED: early stop or deadline reached] ---\n')
                    break

                prompt_ids = encoding.render_conversation_for_completion(conversation, Role.ASSISTANT)
                max_tokens = self.cfg.context_tokens - len(prompt_ids)

                if max_tokens < self.cfg.buffer_tokens:
                    reasoning_parts.append('\n--- [STOPPED: context window exhausted] ---\n')
                    break

                turn_number += 1

                # Create stream request with logprobs enabled for DeepConf
                stream = None
                try:
                    stream = self.client.completions.create(
                        model=self.cfg.served_model_name, 
                        temperature=self.cfg.temperature, 
                        max_tokens=max_tokens, 
                        prompt=prompt_ids, 
                        seed=attempt_seed, 
                        stream=True, 
                        logprobs=self.cfg.top_logprobs,
                        extra_body={
                            'min_p': self.cfg.min_p, 
                            'stop_token_ids': self.stop_token_ids, 
                            'return_token_ids': True
                        },
                        timeout=max(0, deadline - time.time()),
                    )
                except Exception as e:
                    print(f"⚠️ Failed to create completion stream: {e}")
                    reasoning_parts.append(f'\n--- [ERROR: Failed to create stream: {e}] ---\n')
                    break
    
                if stream is None:
                    continue

                try:
                    token_buffer = []
                    text_chunks = []

                    for chunk in stream:
                        if stop_event.is_set() or time.time() > deadline:
                            break

                        new_tokens = chunk.choices[0].token_ids
                        new_text = chunk.choices[0].text

                        if new_tokens:
                            token_buffer.extend(new_tokens)
                            total_tokens += len(new_tokens)
                            text_chunks.append(new_text)

                        # Extract logprobs from chunk for DeepConf confidence
                        if hasattr(chunk.choices[0], 'logprobs') and chunk.choices[0].logprobs is not None:
                            logprobs_data = chunk.choices[0].logprobs
                            if hasattr(logprobs_data, 'top_logprobs') and logprobs_data.top_logprobs:
                                for top_lp in logprobs_data.top_logprobs:
                                    if top_lp:
                                        if isinstance(top_lp, dict):
                                            lp_values = [v.logprob if hasattr(v, 'logprob') else v for v in top_lp.values()]
                                        elif isinstance(top_lp, list):
                                            lp_values = [item.logprob if hasattr(item, 'logprob') else item for item in top_lp]
                                        else:
                                            lp_values = []
                                        if lp_values:
                                            all_logprobs.append(lp_values)
                            elif hasattr(logprobs_data, 'token_logprobs') and logprobs_data.token_logprobs:
                                for lp in logprobs_data.token_logprobs:
                                    if lp is not None:
                                        all_logprobs.append([lp])

                        if '}' in new_text:
                            search_text = ''.join(text_chunks[-self.cfg.search_tokens:])
                            answer = self._scan_for_answer(search_text)

                            if answer is not None:
                                final_answer = answer
                                break

                finally:
                    stream.close()

                # Capture model output text for this turn
                turn_text = ''.join(text_chunks)
                reasoning_parts.append(f'[Turn {turn_number} - Assistant]\n{turn_text}\n')

                if final_answer is not None:
                    break

                if not token_buffer:
                    reasoning_parts.append('\n--- [STOPPED: empty token buffer] ---\n')
                    break

                new_messages = encoding.parse_messages_from_completion_tokens(token_buffer, Role.ASSISTANT)
                conversation.messages.extend(new_messages)
                last_message = new_messages[-1]

                if last_message.channel == 'final':
                    answer_text = last_message.content[0].text
                    final_answer = self._scan_for_answer(answer_text)
                    break

                if last_message.recipient == 'python':
                    python_calls += 1
                    python_code = last_message.content[0].text
                    reasoning_parts.append(f'[Turn {turn_number} - Python Code]\n{python_code}\n')
                    
                    print("🐍 Executing Python code...")
                    tool_responses = local_tool.process_sync_plus(last_message)

                    response_text = tool_responses[0].content[0].text
                    reasoning_parts.append(f'[Turn {turn_number} - Python Output]\n{response_text}\n')

                    if response_text.startswith('[ERROR]') or 'Traceback' in response_text or 'Error:' in response_text:
                        python_errors += 1

                    conversation.messages.extend(tool_responses)

        except Exception as exc:
            python_errors += 1
            reasoning_parts.append(f'\n--- [EXCEPTION: {exc}] ---\n')

        finally:
            if local_tool is not None:
                local_tool.close()

            if sandbox is not None:
                sandbox.reset()
                self.sandbox_pool.put(sandbox)

        # Compute token confidences and group confidences
        token_confs = self._compute_token_confidences(all_logprobs)
        x_positions, group_confs = self._compute_group_confidences(token_confs)
        
        # Use Bottom 10% Group Confidence for weighted voting
        confidence = self._compute_bottom10_confidence(group_confs)
        
        # Join all reasoning parts into a single string
        full_reasoning = '\n'.join(reasoning_parts)
        # Determine if the answer came from \boxed{} (verified) or needs fallback
        verified = final_answer is not None

        # If no \boxed{} answer found, try \unverified_boxed{} as fallback
        if final_answer is None:
            unverified = self._scan_for_unverified_answer(full_reasoning)
            if unverified is not None:
                final_answer = unverified
                print(f"⚠️ Attempt {attempt_index + 1}: No \\boxed{{}} found, using \\unverified_boxed{{{unverified}}} as fallback")

        return {
            'Attempt': attempt_index + 1, 
            'Response Length': total_tokens, 
            'Python Calls': python_calls, 
            'Python Errors': python_errors, 
            'Answer': final_answer,
            'Verified': verified,
            'Confidence': confidence,
            'GroupConfX': x_positions,
            'GroupConfY': group_confs,
            'TokenConfs': token_confs,
            'FullReasoning': full_reasoning
        }

    def _save_token_confidences_csv(
        self, 
        detailed_results: list, 
        ground_truth: int | None, 
        problem_id: int
    ) -> str | None:
        """
        Save raw token confidences to CSV for analysis.
        
        CSV format:
        - problem_id: Problem identifier
        - attempt: Attempt number (1-indexed)
        - answer: The answer produced by this attempt
        - is_correct: Whether the answer matches ground truth (1/0/NA)
        - token_index: Position in the token sequence (0-indexed)
        - token_confidence: Per-token confidence value (C_t)
        
        This allows offline analysis to compare different metrics:
        - Tail confidence (C_tail)
        - Group confidence (C_G)
        - Bottom 10% group confidence (C_bottom-10)
        """
        # Only save during local validation
        if os.getenv('KAGGLE_IS_COMPETITION_RERUN'):
            return None
        
        rows = []
        
        for r in detailed_results:
            token_confs = r.get('TokenConfs', [])
            if not token_confs:
                continue
            
            attempt = r['Attempt']
            answer = r['Answer']
            
            # Determine correctness
            if ground_truth is not None and answer is not None:
                is_correct = 1 if answer == ground_truth else 0
            else:
                is_correct = -1  # Unknown (will be NA in analysis)
            
            for token_idx, conf in enumerate(token_confs):
                rows.append({
                    'problem_id': problem_id,
                    'attempt': attempt,
                    'answer': answer if answer is not None else -1,
                    'is_correct': is_correct,
                    'token_index': token_idx,
                    'token_confidence': round(conf, 6)
                })
        
        if not rows:
            print("⚠️ No token confidence data to save")
            return None
        
        # Create DataFrame and save
        df = pd.DataFrame(rows)
        csv_path = os.path.join(self.cfg.token_conf_dir, f'problem_{problem_id}_token_confs.csv')
        df.to_csv(csv_path, index=False)
        
        print(f"📁 Token confidences saved: {csv_path} ({len(rows)} rows)")
        return csv_path

    def _save_reasoning_csv(
        self,
        detailed_results: list,
        ground_truth: int | None,
        problem_id: int,
        problem_text: str
    ) -> str | None:
        """
        Save full reasoning traces to a CSV for every attempt of this problem.
        
        One CSV per problem, one row per attempt (always 8 rows, including incomplete ones).
        
        Columns:
        - problem_id: Problem identifier
        - problem_text: The original problem statement (truncated for CSV sanity)
        - attempt: Attempt number (1-indexed)
        - answer: The answer produced (or None if unfinished)
        - ground_truth: The correct answer (if known)
        - verdict: 'correct' / 'wrong' / 'unfinished'
        - total_tokens: Total tokens generated in this attempt
        - python_calls: Number of Python tool invocations
        - python_errors: Number of Python errors encountered
        - confidence: C_bottom10 confidence score
        - full_reasoning: The complete multi-turn reasoning trace
        """
        # Only save during local validation
        if os.getenv('KAGGLE_IS_COMPETITION_RERUN'):
            return None
        
        rows = []
        
        # Sort by attempt number so rows are in order
        sorted_results = sorted(detailed_results, key=lambda r: r['Attempt'])
        
        for r in sorted_results:
            answer = r['Answer']
            
            # Determine verdict
            if answer is None:
                verdict = 'unfinished'
            elif ground_truth is not None:
                verdict = 'correct' if answer == ground_truth else 'wrong'
            else:
                verdict = 'unknown'
            
            rows.append({
                'problem_id': problem_id,
                'problem_text': problem_text[:500],  # Truncate for CSV readability
                'attempt': r['Attempt'],
                'answer': answer if answer is not None else '',
                'ground_truth': ground_truth if ground_truth is not None else '',
                'verdict': verdict,
                'total_tokens': r['Response Length'],
                'python_calls': r['Python Calls'],
                'python_errors': r['Python Errors'],
                'confidence': round(r['Confidence'], 6),
                'full_reasoning': r.get('FullReasoning', '')
            })
        
        if not rows:
            print("⚠️ No reasoning data to save")
            return None
        
        df = pd.DataFrame(rows)
        csv_path = os.path.join(self.cfg.reasoning_dir, f'problem_{problem_id}_reasoning.csv')
        df.to_csv(csv_path, index=False)
        
        # Print summary
        verdicts = df['verdict'].value_counts().to_dict()
        verdict_str = ', '.join(f'{v}: {c}' for v, c in verdicts.items())
        print(f"📝 Reasoning traces saved: {csv_path} ({len(rows)} attempts | {verdict_str})")
        return csv_path

    def _plot_confidence_trajectories(
        self, 
        detailed_results: list, 
        ground_truth: int | None, 
        problem_id: int
    ) -> str | None:
        """
        Create a combined plot showing confidence trajectories for all attempts.
        
        Green shades for correct answers, red shades for wrong answers.
        X-axis: Token count (end position of sliding window)
        Y-axis: Group confidence value
        
        Returns the path to the saved plot, or None if not saved.
        """
        # Only plot during local validation
        if os.getenv('KAGGLE_IS_COMPETITION_RERUN'):
            return None
        
        # Filter results with valid group confidence data
        valid_results = [r for r in detailed_results if r['GroupConfX'] and r['GroupConfY']]
        
        if not valid_results:
            print("⚠️ No valid group confidence data to plot")
            return None
        
        # Separate correct and wrong attempts
        correct_attempts = []
        wrong_attempts = []
        unknown_attempts = []
        
        for r in valid_results:
            if ground_truth is not None and r['Answer'] is not None:
                if r['Answer'] == ground_truth:
                    correct_attempts.append(r)
                else:
                    wrong_attempts.append(r)
            else:
                unknown_attempts.append(r)
        
        # Create figure
        fig, ax = plt.subplots(figsize=(14, 8))
        
        # Generate green shades for correct attempts
        if correct_attempts:
            green_cmap = cm.get_cmap('Greens')
            green_shades = [green_cmap(0.4 + 0.5 * i / max(1, len(correct_attempts) - 1)) 
                           for i in range(len(correct_attempts))]
            
            for idx, r in enumerate(correct_attempts):
                ax.plot(
                    r['GroupConfX'], 
                    r['GroupConfY'], 
                    color=green_shades[idx], 
                    linewidth=1.5, 
                    alpha=0.8,
                    label=f"Attempt {r['Attempt']} ✓ (ans={r['Answer']}, C_b10={r['Confidence']:.4f})"
                )
        
        # Generate red shades for wrong attempts
        if wrong_attempts:
            red_cmap = cm.get_cmap('Reds')
            red_shades = [red_cmap(0.4 + 0.5 * i / max(1, len(wrong_attempts) - 1)) 
                         for i in range(len(wrong_attempts))]
            
            for idx, r in enumerate(wrong_attempts):
                ax.plot(
                    r['GroupConfX'], 
                    r['GroupConfY'], 
                    color=red_shades[idx], 
                    linewidth=1.5, 
                    alpha=0.8,
                    label=f"Attempt {r['Attempt']} ✗ (ans={r['Answer']}, C_b10={r['Confidence']:.4f})"
                )
        
        # Gray for unknown (no ground truth)
        if unknown_attempts:
            gray_cmap = cm.get_cmap('Greys')
            gray_shades = [gray_cmap(0.4 + 0.4 * i / max(1, len(unknown_attempts) - 1)) 
                          for i in range(len(unknown_attempts))]
            
            for idx, r in enumerate(unknown_attempts):
                ax.plot(
                    r['GroupConfX'], 
                    r['GroupConfY'], 
                    color=gray_shades[idx], 
                    linewidth=1.5, 
                    alpha=0.8,
                    label=f"Attempt {r['Attempt']} ? (ans={r['Answer']}, C_b10={r['Confidence']:.4f})"
                )
        
        ax.set_xlabel('Token Position (end of window)', fontsize=12)
        ax.set_ylabel('Group Confidence (window avg)', fontsize=12)
        ax.set_title(
            f'Problem {problem_id}: Sliding Window Group Confidence Trajectories\n'
            f'(window={self.cfg.group_window_size}, step={self.cfg.group_step_size}, '
            f'GT={ground_truth})',
            fontsize=14
        )
        ax.legend(loc='upper left', fontsize=9, bbox_to_anchor=(1.02, 1))
        ax.grid(True, alpha=0.3)
        
        plt.tight_layout()
        
        # Save the plot
        plot_path = os.path.join(self.cfg.plot_dir, f'problem_{problem_id}_confidence.png')
        plt.savefig(plot_path, dpi=150, bbox_inches='tight')
        plt.close(fig)
        
        print(f"📊 Confidence plot saved: {plot_path}")
        return plot_path

    def _select_answer(self, detailed_results: list) -> int:
        """
        DeepConf Weighted Majority Voting using Bottom 10% Group Confidence.
        
        Per DeepConf paper:
        1. Filter: Keep only top 90% confident traces (currently disabled)
        2. Weight: Each answer is weighted by its C_bottom-10 score
        3. Vote: Select answer with highest cumulative weighted votes
        
        V(a) = sum(C_bottom-10(t) * I(answer(t) == a)) for t in traces
        """
        # Filter to only traces with valid answers
        valid_results = [r for r in detailed_results if r['Answer'] is not None]
        
        if not valid_results:
            print('\nNo valid answers found.')
            return 0
        
        # Collect weighted votes for each answer
        answer_weights = defaultdict(float)
        answer_counts = defaultdict(int)
        answer_calls = defaultdict(int)

        for result in valid_results:
            answer = result['Answer']
            confidence = result.get('Confidence', 0.0)

            answer_weights[answer] += confidence
            answer_counts[answer] += 1
            answer_calls[answer] += result['Python Calls']

        # Sort by weighted votes (primary), then by count (secondary), then by calls (tertiary)
        sorted_answers = sorted(
            answer_weights.items(), 
            key=lambda item: (item[1], answer_counts[item[0]], answer_calls[item[0]]), 
            reverse=True
        )

        # Display voting results with DeepConf weighted scores
        vote_data = []
        for answer, weight in sorted_answers:
            vote_data.append((
                answer, 
                answer_counts[answer], 
                round(weight, 4),
                answer_calls[answer]
            ))

        vote_dataframe = pd.DataFrame(vote_data, columns=['Answer', 'Votes', 'C_bottom10 Score', 'Calls'])
        display(vote_dataframe)

        final_answer = sorted_answers[0][0]
        final_votes = answer_counts[final_answer]
        final_weight = sorted_answers[0][1]
        final_calls = answer_calls[final_answer]

        print(f'\nFinal Result: {final_answer} | Votes: {final_votes} | C_bottom10 Score: {final_weight:.4f} | Calls: {final_calls}\n')

        return final_answer

    def solve_problem(self, problem: str, ground_truth_answer: int | None = None) -> int:
        
        problem_start_time = time.time()
        self.problem_counter += 1
        problem_id = self.problem_counter
        
        print(f'\nProblem {problem_id}: {problem[:200]}...\n')

        user_input = f'{problem} {self.cfg.preference_prompt}'
        elapsed_global = time.time() - self.notebook_start_time
        time_left = self.cfg.notebook_limit - elapsed_global
        problems_left_others = max(0, self.problems_remaining - 1)
        reserved_time = problems_left_others * self.cfg.base_problem_timeout

        budget = time_left - reserved_time
        budget = min(budget, self.cfg.high_problem_timeout)
        budget = max(budget, self.cfg.base_problem_timeout)

        deadline = time.time() + budget

        print(f'Budget: {budget:.2f} seconds | Deadline: {deadline:.2f}\n')

        tasks = []

        for attempt_index in range(self.cfg.attempts):
            tasks.append((self.cfg.system_prompt, attempt_index))

        detailed_results = []
        valid_answers = []

        stop_event = threading.Event()

        executor = ThreadPoolExecutor(max_workers=self.cfg.workers)

        try:
            futures = []

            for (system_prompt, attempt_index) in tasks:
                future = executor.submit(
                    self._process_attempt, 
                    user_input, 
                    system_prompt, 
                    attempt_index, 
                    stop_event, 
                    deadline
                )

                futures.append(future)

            for future in as_completed(futures):
                try:
                    result = future.result()
                    detailed_results.append(result)

                    if result['Answer'] is not None:
                        valid_answers.append(result['Answer'])

                    counts = Counter(valid_answers).most_common(1)

                    if counts and counts[0][1] >= self.cfg.early_stop:
                        stop_event.set()

                        for f in futures:
                            f.cancel()

                        break

                except Exception as exc:
                    print(f'Future failed: {exc}')
                    continue

        finally:
            executor.shutdown(wait=False, cancel_futures=True)
            self.problems_remaining = max(0, self.problems_remaining - 1)

        # Print the inference time and budget
        used_time = time.time() - problem_start_time
        saved_time = max(0.0, budget - used_time)
        print(f"[Budget]: {budget:.2f}s\n")
        print(f"[Inference] Took {used_time:.2f}s\n")
        print(f"[Saved time]: {saved_time:.2f}s\n")

        if detailed_results:
            # Prepare display dataframe (without GroupConf, TokenConf, and FullReasoning columns)
            display_results = []
            for r in detailed_results:
                display_results.append({
                    'Attempt': r['Attempt'],
                    'Answer': r['Answer'],
                    'C_bottom10': round(r['Confidence'], 4),
                    'Response Length': r['Response Length'],
                    'Python Calls': r['Python Calls'],
                    'Python Errors': r['Python Errors']
                })
            
            results_dataframe = pd.DataFrame(display_results)
            results_dataframe['Answer'] = results_dataframe['Answer'].astype('Int64')
            display(results_dataframe)

        if not valid_answers:
            print('\nResult: 0\n')
            # Still save reasoning traces even when no valid answers
            self._save_reasoning_csv(
                detailed_results,
                ground_truth_answer,
                problem_id,
                problem
            )
            return 0

        final_answer = self._select_answer(detailed_results)
        
        # Save full reasoning traces to CSV (only during local validation)
        self._save_reasoning_csv(
            detailed_results,
            ground_truth_answer,
            problem_id,
            problem
        )
        
        # Save token confidences to CSV (only during local validation)
        self._save_token_confidences_csv(
            detailed_results, 
            ground_truth_answer, 
            problem_id
        )
        
        # Plot confidence trajectories (only during local validation)
        plot_path = self._plot_confidence_trajectories(
            detailed_results, 
            ground_truth_answer, 
            problem_id
        )
        
        # Display the plot inline if saved
        if plot_path and os.path.exists(plot_path):
            from IPython.display import Image, display as ipy_display
            print("\n📈 Confidence Trajectory Plot:")
            ipy_display(Image(filename=plot_path))

        return final_answer

    def __del__(self):

        if hasattr(self, 'server_process'):
            self.server_process.terminate()
            self.server_process.wait()

        if hasattr(self, 'log_file'):
            self.log_file.close()

        if hasattr(self, 'sandbox_pool'):
            while not self.sandbox_pool.empty():
                try:
                    sb = self.sandbox_pool.get_nowait()
                    sb.close()

                except Exception:
                    pass

In [ ]:
solver = AIMO3Solver(CFG)

In [ ]:
def predict(id_: pl.DataFrame, question: pl.DataFrame, answer: Optional[pl.DataFrame] = None) -> pl.DataFrame:
    global correct_count, total_count, predictions
    
    question_id = id_.item(0)
    question_text = question.item(0)
    
    print("------")
    print(f"ID: {question_id}")
    print(f"Question: {question_text[:200]}...")
    
    # Get ground truth for plotting (only available in local validation)
    gt_answer = ground_truth.get(question_id, None)
    
    final_answer = solver.solve_problem(question_text, ground_truth_answer=gt_answer)
    predictions[question_id] = final_answer

    # Check accuracy if ground truth available
    total_count += 1
    if question_id in ground_truth:
        gt = ground_truth[question_id]
        is_correct = (final_answer == gt)
        if is_correct:
            correct_count += 1
        status = "✅" if is_correct else "❌"
        print(f"Answer: {final_answer} | Ground Truth: {gt} | {status}")
        print(f"📊 Running Accuracy: {correct_count}/{total_count} ({100*correct_count/total_count:.1f}%)")
    else:
        print(f"Answer: {final_answer}")
    
    print("------\n")
    
    return pl.DataFrame({'id': question_id, 'answer': final_answer})

In [ ]:
# Load reference data and keep ground truth for accuracy calculation
df = pd.read_csv(
    "/kaggle/input/ai-mathematical-olympiad-progress-prize-3/test.csv"
)

# Store ground truth answers for accuracy calculation (only in local mode)
ground_truth = dict(zip(df["id"], df["answer"])) if "answer" in df.columns else {}

# Create input file without answers
df.drop("answer", axis=1, errors="ignore").to_csv("reference.csv", index=False)

# Track predictions for accuracy calculation
predictions = {}
correct_count = 0
total_count = 0

In [ ]:
# # Load reference data and keep ground truth for accuracy calculation
# df = pd.read_csv(
#     "/kaggle/input/omni-math-hardestdifficulty-9/omni_math_hard.csv"
# )

# # df = df[df['id']==12].copy()

# df = df[['id','problem','answer']]

# # Store ground truth answers for accuracy calculation (only in local mode)
# ground_truth = dict(zip(df["id"], df["answer"])) if "answer" in df.columns else {}

# # Create input file without answers
# df.drop("answer", axis=1, errors="ignore").to_csv("reference.csv", index=False)

# # Track predictions for accuracy calculation
# predictions = {}
# correct_count = 0
# total_count = 0

# print(f"Dataset prepared with {len(df)} problems.")

In [ ]:
inference_server = kaggle_evaluation.aimo_3_inference_server.AIMO3InferenceServer(predict)

if os.getenv('KAGGLE_IS_COMPETITION_RERUN'):
    inference_server.serve()
    
else:
    inference_server.run_local_gateway(("reference.csv",))
    #inference_server.run_local_gateway(
    #    ('/kaggle/input/ai-mathematical-olympiad-progress-prize-3/test.csv',)
    #)